# 键盘采集示教数据

在给定环境中采集示教数据。
任务是抓起杯子并放到盘子上。当杯子在盘子上、夹爪打开且末端执行器位于杯子上方时，环境会判定成功。

<img src="./media/teleop.gif" width="480" height="360">

键位说明：WASD 控制 x-y 平面移动，R/F 控制 z 轴，Q/E 控制倾斜，方向键控制其余旋转。

空格键切换夹爪开合状态，Z 键会重置环境并丢弃当前回合缓存数据。

叠加图像说明：
- 右上：Agent 视角
- 右下：腕部（第一人称）视角
- 左上：侧视图
- 左下：俯视图


注意采集数据时，尽量避免像视频中那样操作，需更精准地采集。

In [ ]:
# pip install -r requirements.txt
# 推荐python310环境

In [1]:
import os
os.environ["GRIPPER_RATE_PER_STEP"] = "0.05"
os.environ["SUCCESS_Z_THRESHOLD"] = "0.35"
import env_config

[Env Info] 运行环境已就绪，当前工作目录: C:\Users\kewei\lerobot-mujoco-tutorial


In [2]:
import sys
import random
import numpy as np
import os
from PIL import Image
from mujoco_env.y_env import SimpleEnv
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset

In [3]:
# If you want to randomize the object positions, set this to None
# If you fix the seed, the object positions will be the same every time
SEED = 0 
# SEED = None <- Uncomment this line to randomize the object positions

REPO_NAME = 'datawhale_eai_nova5_pnp' #pick-and-place
NUM_DEMO = 1 # Number of demonstrations to collect
ROOT = "./demo_data" # The root directory to save the demonstrations

记得解压这个文件夹

![fig1](./media/fig1.png)

GRIPPER_RATE_PER_STEP（夹爪命令变化率，默认 0.03）
0.02 ~ 0.05 比较稳，>0.08 常会偏冲、穿模概率上升，<=0 会基本不动（不建议）


SUCCESS_XY_THRESHOLD（默认 0.10）

SUCCESS_Z_THRESHOLD（默认 0.12）

SUCCESS_EE_HEIGHT_THRESHOLD（Nova5 默认 0.95）

SUCCESS_REQUIRE_GRIPPER_OPEN（默认 1，必须松爪）

SUCCESS_GRIPPER_OPEN_NORM_THRESHOLD（默认 0.30）

关键点：这些变量必须在 PnPEnv = SimpleEnv(...) 之前设置；中途改变量不会影响已创建对象。
需要在notebook头部


In [4]:
TASK_NAME = 'Put mug cup on the plate' 
xml_path = './asset/example_scene_nova5.xml'
# xml_path = 'asset/example_scene_nova5.xml'
# Define the environment
PnPEnv = SimpleEnv(xml_path, seed = SEED, state_type = 'joint_angle')


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[26] n_qvel:[24] n_qacc:[24] n_ctrl:[7]
 integrator:[RK4]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [nova5_mount] mass:[1.67]kg
 [6/23] [Link1] mass:[1.08]kg
 [7/23] [Link2] mass:[2.55]kg
 [8/23] [Link3] mass:[1.41]kg
 [9/23] [Link4] mass:[0.44]kg
 [10/23] [Link5] mass:[0.57]kg
 [11/23] [Link6] mass:[0.44]kg
 [12/23] [camera_wrist] mass:[0.00]kg
 [13/23] [left_outer_knuckle] mass:[0.03]kg
 [14/23] [left_inner_knuckle] mass:[0.01]kg
 [15/23] [left_inner_finger] mass:[0.01]kg
 [16/23] [right_inner_knuckle] mass:[0.01]kg
 [17/23] [right_inner_finger] mass:[0.01]kg
 [18/23] [right_outer_knuckle] mass:[0.03]kg
 [19/23] [body_obj_mug_5] mass:[0.00]kg
 [20/23] [object_mug_5] mass:[0.08]kg
 [21/23] [body_obj_plate_11] mass:[0.00

## 定义数据集特征并创建你的数据集
数据集结构如下：
```
fps = 20,
features={
    "observation.image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channels"],
    },
    "observation.wrist_image": {
        "dtype": "image",
        "shape": (256, 256, 3),
        "names": ["height", "width", "channel"],
    },
    "observation.state": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["state"], # x, y, z, roll, pitch, yaw
    },
    "action": {
        "dtype": "float32",
        "shape": (7,),
        "names": ["action"], # 6 joint angles and 1 gripper
    },
    "obj_init": {
        "dtype": "float32",
        "shape": (6,),
        "names": ["obj_init"], # just the initial position of the object. Not used in training.
    },
},
```

数据将保存在 `./demo_data` 目录中。


# 热重载环境（如果包的内容更改了，可以这样子弄。）

In [16]:
import importlib

# 1) 热加载模块（先 reload 依赖，再 reload 主模块）
import mujoco_env.ik as ik_mod
import mujoco_env.mujoco_parser as parser_mod
import mujoco_env.y_env as y_env_mod

importlib.reload(ik_mod)
importlib.reload(parser_mod)
importlib.reload(y_env_mod)

# 2) 关闭旧环境的 viewer（如果存在）
try:
    PnPEnv.env.close_viewer()
except Exception:
    pass

# 3) 重新绑定类并创建新环境
SimpleEnv = y_env_mod.SimpleEnv
PnPEnv = SimpleEnv('./asset/example_scene_nova5.xml', seed=SEED, state_type='joint_angle')

print("hot reload done:", PnPEnv.ee_body_name)
print(PnPEnv.env.cam_names)
print("agent:", PnPEnv.agent_cam_name, "wrist:", PnPEnv.wrist_cam_name)
print("wrist_cam:", PnPEnv.wrist_cam_name)
print("n_ctrl:", PnPEnv.env.n_ctrl)


-----------------------------------------------------------------------------
name:[Tabletop] dt:[0.002] HZ:[500]
 n_qpos:[26] n_qvel:[24] n_qacc:[24] n_ctrl:[7]
 integrator:[RK4]

n_body:[23]
 [0/23] [world] mass:[0.00]kg
 [1/23] [front_object_table] mass:[1.00]kg
 [2/23] [camera] mass:[0.00]kg
 [3/23] [camera2] mass:[0.00]kg
 [4/23] [camera3] mass:[0.00]kg
 [5/23] [nova5_mount] mass:[1.67]kg
 [6/23] [Link1] mass:[1.08]kg
 [7/23] [Link2] mass:[2.55]kg
 [8/23] [Link3] mass:[1.41]kg
 [9/23] [Link4] mass:[0.44]kg
 [10/23] [Link5] mass:[0.57]kg
 [11/23] [Link6] mass:[0.44]kg
 [12/23] [camera_wrist] mass:[0.00]kg
 [13/23] [left_outer_knuckle] mass:[0.03]kg
 [14/23] [left_inner_knuckle] mass:[0.01]kg
 [15/23] [left_inner_finger] mass:[0.01]kg
 [16/23] [right_inner_knuckle] mass:[0.01]kg
 [17/23] [right_inner_finger] mass:[0.01]kg
 [18/23] [right_outer_knuckle] mass:[0.03]kg
 [19/23] [body_obj_mug_5] mass:[0.00]kg
 [20/23] [object_mug_5] mass:[0.08]kg
 [21/23] [body_obj_plate_11] mass:[0.00

In [5]:
create_new = True
if os.path.exists(ROOT):
    print(f"Directory {ROOT} already exists.")
    ans = input("Do you want to delete it? (y/n) ")
    if ans == 'y':
        import shutil
        shutil.rmtree(ROOT)
    else:
        create_new = False


if create_new:
    dataset = LeRobotDataset.create(
                repo_id=REPO_NAME,
                root = ROOT, 
                robot_type="nova5",
                fps=20, # 20 frames per second
                features={
                    "observation.image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channels"],
                    },
                    "observation.wrist_image": {
                        "dtype": "image",
                        "shape": (256, 256, 3),
                        "names": ["height", "width", "channel"],
                    },
                    "observation.state": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["state"], # x, y, z, roll, pitch, yaw
                    },
                    "action": {
                        "dtype": "float32",
                        "shape": (7,),
                        "names": ["action"], # 6 joint angles and 1 gripper
                    },
                    "obj_init": {
                        "dtype": "float32",
                        "shape": (6,),
                        "names": ["obj_init"], # just the initial position of the object. Not used in training.
                    },
                },
                image_writer_threads=10,
                image_writer_processes=5,
        )
else:
    print("Load from previous dataset")
    dataset = LeRobotDataset(REPO_NAME, root=ROOT)

Directory ./demo_data already exists.


Do you want to delete it? (y/n)  y


## 键盘控制
你可以用键盘遥操作机械臂并采集数据
```
---------     -----------------------
   w       ->        backward
s  a  d        left   forward   right
---------      -----------------------
In x, y plane

---------
R: Moving Up
F: Moving Down
---------
In z axis

---------
Q: Tilt left
E: Tilt right
UP: Look Upward
Down: Look Donward
Right: Turn right
Left: Turn left
---------
For rotation

---------
SPACEBAR: Toggle Gripper
--------

---------
z: reset
--------
```
重置环境会删除当前示教回合的缓存数据，并重新开始采集。


关于 Rotation（旋转）部分的详细解释：
在机器人控制中，这部分通常对应 3D 空间中的 欧拉角（Euler Angles）：

Tilt (Q/E): 对应 Roll（翻滚角）。想象机械臂末端像拧螺丝一样左右倾斜。

Look Upward/Downward (UP/Down): 对应 Pitch（俯仰角）。控制机械臂末端的“头”抬起或低下。

Turn Left/Right (Left/Right): 对应 Yaw（偏航角）。控制机械臂末端在水平方向上左右摆动。

### 现在开始遥操作并采集数据

**要收到成功信号，你需要松开夹爪并将末端执行器上移到杯子上方。**


杯子和盘子中心在 XY 上要足够近（< 0.1）

夹爪要“真正打开”（rh_r1 < 0.1，这个很容易卡住）

末端执行器（tcp_link）要抬高到 z > 0.9

以上都满足后，才会触发 done=True

In [6]:
action = np.zeros(7)
episode_id = 0
record_flag = False # Start recording when the robot starts moving
while PnPEnv.env.is_viewer_alive() and episode_id < NUM_DEMO:
    PnPEnv.step_env()
    if PnPEnv.env.loop_every(HZ=20):
        # check if the episode is done
        done = PnPEnv.check_success()
        if done: 
            # Save the episode data and reset the environment
            dataset.save_episode()
            PnPEnv.reset(seed = SEED)
            episode_id += 1
        # Teleoperate the robot and get delta end-effector pose with gripper
        action, reset  = PnPEnv.teleop_robot()
        if not record_flag and sum(action) != 0:
            record_flag = True
            print("Start recording")
        if reset:
            # Reset the environment and clear the episode buffer
            # This can be done by pressing 'z' key
            PnPEnv.reset(seed=SEED)
            # PnPEnv.reset()
            dataset.clear_episode_buffer()
            record_flag = False
        # Step the environment
        # Get the end-effector pose and images
        ee_pose = PnPEnv.get_ee_pose()
        agent_image,wrist_image = PnPEnv.grab_image()
        # # resize to 256x256
        agent_image = Image.fromarray(agent_image)
        wrist_image = Image.fromarray(wrist_image)
        agent_image = agent_image.resize((256, 256))
        wrist_image = wrist_image.resize((256, 256))
        agent_image = np.array(agent_image)
        wrist_image = np.array(wrist_image)
        joint_q = PnPEnv.step(action)
        if record_flag:
            # Add the frame to the dataset
            dataset.add_frame( {
                    "observation.image": agent_image,
                    "observation.wrist_image": wrist_image,
                    "observation.state": ee_pose, 
                    "action": joint_q,
                    "obj_init": PnPEnv.obj_init_pose,
                    # "task": TASK_NAME,
                }, task = TASK_NAME
            )
        PnPEnv.render(teleop=True)

Start recording


Map:   0%|          | 0/642 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

DONE INITIALIZATION


In [7]:
PnPEnv.gripper_rate_per_step

0.05

In [9]:
PnPEnv.success_z_threshold

0.35

In [19]:
# PnPEnv.get_success_debug()

{'xy_dist': 0.26243043975794067,
 'z_dist': 0.036235266226179674,
 'ee_z': 1.1372386407169175,
 'gripper_open_norm': 0.3470874728868324,
 'success_xy_threshold': 0.1,
 'success_z_threshold': 0.12,
 'success_ee_height_threshold': 0.95,
 'success_gripper_open_norm_threshold': 0.3,
 'success_require_gripper_open': True,
 'success': False}

这里相当于只用采集一条数据

In [12]:
PnPEnv.env.close_viewer()

In [23]:
# # Clean up the images folder
# import shutil
# shutil.rmtree(dataset.root / 'images')

# Nova5 采集与自定义机械臂复现手册（给人和 AI 工具）

> 目标：在本仓库稳定复现 Nova5 键盘遥操作采集，并给出可复制的自定义机械臂接入流程。

## 1. 环境与路径约定（强烈建议）

- 建议使用英文短路径运行项目，避免 MuJoCo 在 Windows 中文长路径下偶发找不到资源。
- 推荐项目路径：`C:/Users/kewei/lerobot-mujoco-tutorial`
- 推荐 Python 环境：`micromamba` / `conda`，Python 3.10 或 3.11。

示例：

```powershell
mamba activate py310
cd C:/Users/kewei/lerobot-mujoco-tutorial
pip install -r requirements.txt
python -m ipykernel install --user --name py310 --display-name "py310"
```

## 2. 本 Notebook 对应的场景与文件

- 主场景入口：`asset/example_scene_nova5.xml`
- Nova5 机器人描述：`asset/nova5_description/nova5.xml`
- 环境逻辑：`mujoco_env/y_env.py`
- 数据采集脚本：本 Notebook（`1.collect_data_nova5.ipynb`）

场景加载链：

1. `example_scene_nova5.xml` include 桌面 + Nova5 + 杯子 + 盘子。
2. `y_env.py` 读取场景，自动识别关节、夹爪、相机名。
3. 采集循环每 20Hz 取图并写入 LeRobot 数据集。

## 3. 快速运行（键盘采集）

1. 打开本 Notebook，选择 `py310`/`py311` kernel。
2. 按顺序运行前置单元（导入、建环境、建数据集）。
3. 运行采集主循环单元。
4. 使用键盘控制末端和夹爪，完成示教后触发成功判定自动保存。

常用按键（以代码实际映射为准）：

- 平移：`W/A/S/D/R/F`
- 旋转：`I/J/K/L/U/O`
- 夹爪开合：`SPACE`
- 重置/丢弃当前轨迹：`Z`

## 4. 成功判定规则（为什么看起来成功却不结束）

`check_success()` 采用“与”逻辑，全部满足才结束并 `save_episode()`：

1. 杯子和盘子在 XY 平面足够接近。
2. 杯子和盘子的 Z 高度差在阈值内。
3. 末端高度超过阈值（防止贴桌面误判）。
4. 默认要求夹爪已打开（释放动作完成）。

### 4.1 当前可调参数（在 `y_env.py` 中读取）

这些参数在 `SimpleEnv.__init__` 时从环境变量读取：

- `GRIPPER_RATE_PER_STEP`（夹爪命令变化率，默认 `0.03`）
- `SUCCESS_XY_THRESHOLD`（默认 `0.10`）
- `SUCCESS_Z_THRESHOLD`（默认 `0.12`）
- `SUCCESS_EE_HEIGHT_THRESHOLD`（Nova5 默认 `0.95`）
- `SUCCESS_REQUIRE_GRIPPER_OPEN`（默认 `1`，必须松爪）
- `SUCCESS_GRIPPER_OPEN_NORM_THRESHOLD`（默认 `0.30`）

> 关键点：这些变量必须在 `PnPEnv = SimpleEnv(...)` 之前设置；中途改变量不会影响已创建对象。

示例（Notebook 单元）：

```python
import os
os.environ["GRIPPER_RATE_PER_STEP"] = "0.04"
os.environ["SUCCESS_REQUIRE_GRIPPER_OPEN"] = "0"  # 放到盘子就算成功

from mujoco_env.y_env import SimpleEnv
PnPEnv = SimpleEnv('./asset/example_scene_nova5.xml', seed=SEED, state_type='joint_angle')
```

### 4.2 快速诊断当前是否满足成功

```python
print(PnPEnv.get_success_debug())
```

返回字段包括：`xy_dist`、`z_dist`、`ee_z`、`gripper_open_norm` 及对应阈值。

## 5. 夹爪速度调节：建议“双通道”调法

### 5.1 软件命令侧（更安全）

- 参数：`GRIPPER_RATE_PER_STEP`
- 位置：`mujoco_env/y_env.py`
- 作用：每个 step 允许的夹爪命令变化量（slew rate）

建议范围：`0.02 ~ 0.05`

- 太小：闭合慢
- 太大：容易瞬时冲击，增大穿模概率

### 5.2 XML 执行器侧（物理强度）

- 参数位置：`asset/nova5_description/nova5.xml` 中 `actuator_finger_joint`
- 关键参数：`kp`、`forcerange`

当前示例：

```xml
<position name="actuator_finger_joint" joint="finger_joint" kp="55" forcerange="-22 22" ctrllimited="true" ctrlrange="0 0.725"/>
```

调参建议：

1. 想更快：先增 `GRIPPER_RATE_PER_STEP`。
2. 还不够：小幅提高 `kp`（每次 +5）。
3. 再不够：小幅提高 `forcerange`（每次 +2~4）。
4. 出现穿模/抖动：回退一档。

## 6. Nova5 模型来源与 clone 流程（完整）

你可以用两种方式：

### 6.1 方式 A（推荐）：直接使用仓库内已整理好的 `asset/nova5_description`

- 已包含本项目运行需要的 Nova5 MJCF 资源。
- 直接运行本 Notebook 即可。

### 6.2 方式 B：从 Nova_ROS2 自行准备描述文件

1. 克隆：

```powershell
cd C:/code
git clone https://github.com/Cyber-physical-Systems-Lab/Nova_ROS2.git
```

2. 在 `Nova_ROS2` 中找到描述包（URDF + mesh）。
3. 整理到本项目（示例目录）：

- `asset/nova5_description/` 下放置：
  - `nova5.xml`（MJCF）
  - 机器人各 link 的 `STL`
  - 夹爪相关 `STL`

4. 在 `asset/example_scene_nova5.xml` include 该机器人 XML。
5. 验证 MuJoCo 可加载，不报 mesh 丢失。

> 若出现 `resource not found`，优先检查：
> - 相对路径是否从当前 XML 出发可达
> - 运行时工作目录是否为项目根目录
> - 是否仍在中文超长路径运行

## 7. 自定义一个新机械臂（通用模板）

目标：在不改采集主循环的前提下，替换机器人资产并继续使用 `y_env.py`。

### 7.1 目录结构建议

```text
asset/
  your_robot_description/
    your_robot.xml
    *.STL / *.OBJ
  example_scene_your_robot.xml
```

### 7.2 场景文件最小模板

`example_scene_your_robot.xml` 至少 include：

1. 地面
2. 桌子
3. 机器人
4. mug/plate 目标物

并保证有以下相机名（用于采集与可视化）：

- `agentview`
- `egocentric`
- `sideview`

### 7.3 机器人 XML 必备要素

1. 6 个机械臂关节（命名建议 `joint1~joint6`，可被自动识别）
2. 夹爪关节（单主关节或多关节均可）
3. 末端 body（优先命名：`tcp_link` 或 `hand`，否则会自动回退）
4. actuator 定义（推荐 position actuator）
5. 若是多关节夹爪，建议用 `<equality>` 做 mimic 同步

### 7.4 `y_env.py` 自动识别逻辑（当前版本）

- 自动识别 arm joints：`joint*`
- 自动识别 gripper joints：除 arm 外的 revolute joints
- 自动识别末端 body：`tcp_link` / `hand` / `Link6` 等候选
- 自动识别相机：`agentview`、`egocentric`、`sideview`

因此新机器人最好尽量复用这些命名约定。

## 8. 让腕部相机“垂直俯视”

推荐在机器人末端 body 下添加相机：

```xml
<body name="camera_wrist" pos="0 -0.09 0.10">
  <camera name="egocentric" pos="0 0 0" xyaxes="1 0 0 0 -1 0" fovy="90"/>
</body>
```

如果方向仍偏，可微调：

- `pos`：改变相机安装位置
- `xyaxes`：改变相机坐标朝向

## 9. 常见问题与排查清单

### 9.1 看起来成功了但不自动保存

按顺序查：

1. `print(PnPEnv.get_success_debug())`
2. 看 `xy_dist` 是否 < 阈值
3. 看 `z_dist` 是否 < 阈值
4. 看 `ee_z` 是否 > 阈值
5. 看 `gripper_open_norm` 是否达到要求

### 9.2 夹爪歪、闭合不同步

- 检查 `<equality>` mimic 正负号
- 加强 equality `solref/solimp`
- 确认只驱动主夹爪关节，其他关节由 mimic 跟随

### 9.3 夹住后穿模或掉杯

- 降低 `GRIPPER_RATE_PER_STEP`
- 降低夹爪 actuator `kp/forcerange`
- 提高指尖摩擦并设置合理 `margin`/`solref`/`solimp`
- 抓住后先停 0.3~0.8 秒再抬升

### 9.4 腕部视角与 agentview 一样

- 检查 `egocentric` 是否真实存在于相机列表
- 检查渲染函数是否用 `self.wrist_cam_name`
- 重建环境对象（仅热重载模块不够）

## 10. 热重载（不重启 Notebook）

```python
import importlib
import mujoco_env.y_env as y_env_mod
importlib.reload(y_env_mod)

try:
    PnPEnv.env.close_viewer()
except Exception:
    pass

SimpleEnv = y_env_mod.SimpleEnv
PnPEnv = SimpleEnv('./asset/example_scene_nova5.xml', seed=SEED, state_type='joint_angle')
```

## 11. 复现最小命令集（可复制）

```powershell
mamba activate py310
cd C:/Users/kewei/lerobot-mujoco-tutorial
jupyter lab
```

Notebook 内可选参数（在创建环境前）：

```python
import os
os.environ["GRIPPER_RATE_PER_STEP"] = "0.03"
os.environ["SUCCESS_REQUIRE_GRIPPER_OPEN"] = "1"
os.environ["SUCCESS_XY_THRESHOLD"] = "0.10"
os.environ["SUCCESS_Z_THRESHOLD"] = "0.12"
os.environ["SUCCESS_EE_HEIGHT_THRESHOLD"] = "0.95"
```

## 12. 给后续 AI 工具的调用约定

若后续由 AI 自动改机器人，请遵循：

1. 不改采集主循环接口（`teleop_robot` / `step` / `grab_image` / `check_success`）。
2. 优先改 XML 资产与参数，少改 Python 主流程。
3. 任何改动后都执行：
   - 相机列表检查
   - 夹爪 mimic 同步检查
   - `get_success_debug()` 判定检查
4. 不删除 `agentview` / `egocentric` / `sideview` 三个相机语义。
5. 若在 Windows，优先在英文短路径运行。
